# Goal
Compute proxy visual features that roughly correspond to dermatological concepts like:
- Asymmetry
- Border irregularity
- Color variation
- Diameter / size


| Concept             | Automatic proxy                   | What it reveals    |
| ------------------- | --------------------------------- | ------------------ |
| Asymmetry           | Difference between flipped halves | Shape imbalance    |
| Border irregularity | Boundary curvature variance       | Jagged edges       |
| Color variation     | Delta-E (color deviation)         | Pigment unevenness |
| Diameter            | Radial distance map               | Lesion size cue    |

## Extension Idea: Pseudo-Concept Maps for Feature-Level Explainability

To move beyond coarse lesion-level explainability, we propose generating pseudo-concept maps that approximate dermatological features used in clinical diagnosis, following the ABCD (Asymmetry, Border, Color, Diameter) rule for melanoma assessment. These maps are computed automatically using image processing techniques, without requiring expert annotations.

## Effort
| Task                                                    | Tools                     | Effort         |
| ------------------------------------------------------- | ------------------------- | -------------- |
| Use existing mask                                       | NA                        | 🟢 Easy        |
| Generate pseudo-mask (thresholding or pretrained model) | OpenCV / pretrained U-Net | 🟢 Easy–Medium |
| Compute asymmetry (flip and compare halves)             | NumPy operations          | 🟢 Easy        |
| Compute border irregularity (curvature of contour)      | OpenCV contours           | 🟡 Medium      |
| Compute color variation (delta-E in LAB)                | OpenCV / skimage.color    | 🟢 Easy        |
| Compute diameter (distance transform)                   | NumPy / skimage.measure   | 🟢 Easy        |


## A. Asymmetry map

Idea: Malignant lesions are often asymmetric.

How:
- Compute lesion mask centroid.
- Rotate and flip the lesion across its vertical/horizontal axes.
- Subtract mirrored halves -> the absolute difference gives a asymmetry heatmap (highlighting irregular sides).

In [ ]:
import cv2, numpy as np
flipped = cv2.flip(mask, 1)
diff = cv2.absdiff(mask, flipped)
asymmetry_map = diff / diff.max()


## helper function to convert mask to binary
def to_binary_mask(mask):
    if mask.dtype == np.bool_:
        return mask.astype(np.uint8)
    elif mask.dtype == np.uint8:
        return (mask > 0).astype(np.uint8)
    else:
        raise ValueError("Unsupported mask dtype: {}".format(mask.dtype))

## robust version
def compute_asymmetry_map(mask):
    m = to_binary_mask(mask)
    ys, xs = np.where(m > 0)
    if len(xs) == 0:
        return np.zeros_like(m, dtype=np.float32)

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    roi = m[y0:y1+1, x0:x1+1]

    flipped = cv2.flip(roi, 1)  # horizontal flip
    diff = cv2.absdiff(roi, flipped).astype(np.float32)

    # Put back into full image coordinates
    out = np.zeros_like(m, dtype=np.float32)
    out[y0:y1+1, x0:x1+1] = diff

    # keep only inside lesion
    out *= m
    mx = out.max()
    return out / (mx + 1e-12)

## B. Border irregularity map

Idea: Jagged, uneven borders often indicate malignancy.

How:

- Find the contour from the mask.
- Compute curvature or local variance of distance from centroid to boundary points.
- Mark boundary points with high curvature or high variance as irregular.
- Spread that information slightly inward (dilate 3–5 pixels) to create a border irregularity map.

Result: A thin rim around lesion showing irregular edges.


### other toughts:
A practical, reproducible proxy is: radial distance variability along the boundary.

Extract contour points

Compute distances from centroid -> 1D “radius signal”

Smooth it and measure deviation/roughness

Paint boundary pixels with intensity proportional to irregularity

Dilate slightly inward to create a rim map

In [ ]:
def border_irregularity_map(mask, smooth_ksize=21, z_thresh=1.5, dilate_px=5):
    m = to_binary_mask(mask)

    # contour
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros_like(m, dtype=np.float32)

    cnt = max(contours, key=cv2.contourArea)  # largest component
    pts = cnt[:, 0, :]  # (N,2) [x,y]

    # centroid
    Mmom = cv2.moments(m)
    if Mmom["m00"] == 0:
        return np.zeros_like(m, dtype=np.float32)
    cx = Mmom["m10"] / Mmom["m00"]
    cy = Mmom["m01"] / Mmom["m00"]

    # radial distance signal r[t]
    dx = pts[:, 0].astype(np.float32) - cx
    dy = pts[:, 1].astype(np.float32) - cy
    r = np.sqrt(dx*dx + dy*dy)  # length N

    # smooth r to get "expected" radius; use 1D blur via convolution
    k = max(3, smooth_ksize | 1)  # ensure odd
    kernel = np.ones(k, dtype=np.float32) / k
    r_smooth = np.convolve(np.r_[r[-k//2:], r, r[:k//2]], kernel, mode="valid")  # wrap-around smoothing

    # deviation from smooth boundary (roughness)
    dev = np.abs(r - r_smooth)

    # robust normalize using z-score-like scaling
    med = np.median(dev)
    mad = np.median(np.abs(dev - med)) + 1e-12
    z = (dev - med) / (1.4826 * mad)  # robust z
    z = np.clip(z, 0, None)

    # threshold + scale to [0,1]
    score = np.clip((z - z_thresh) / (z.max() - z_thresh + 1e-12), 0, 1)

    # paint boundary scores into an image
    out = np.zeros_like(m, dtype=np.float32)
    for (x, y), s in zip(pts, score):
        out[int(y), int(x)] = max(out[int(y), int(x)], float(s))

    # thicken border and keep it inside lesion
    if dilate_px > 0:
        k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*dilate_px+1, 2*dilate_px+1))
        out = cv2.dilate(out, k2)

    out *= m
    mx = out.max()
    return out / (mx + 1e-12)

## C. Color variation map

Idea: Malignant lesions have uneven color distributions.
How:

Convert lesion region to LAB or HSV color space.

Compute per-pixel color difference from lesion mean ($\Delta E $ in LAB).

Normalise -> pixels with higher $\Delta E $ = higher color variance.

In [ ]:
def color_variation_map(img_rgb, mask):
    m = to_binary_mask(mask).astype(bool)
    if m.sum() == 0:
        return np.zeros(mask.shape, dtype=np.float32)

    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, a, b = lab[...,0], lab[...,1], lab[...,2]

    # mean colour inside lesion
    Lm, am, bm = L[m].mean(), a[m].mean(), b[m].mean()

    deltaE = np.sqrt((L - Lm)**2 + (a - am)**2 + (b - bm)**2)
    deltaE *= m.astype(np.float32)

    mx = deltaE.max()
    return deltaE / (mx + 1e-12)

## D. Diameter map (size cue)

Idea: Larger lesions (>6 mm) are riskier.

How:

- Use mask area and pixel scale (if known) -> scalar feature (not spatial map).

- Or make a radial map from lesion center: each pixel value = distance from centroid normalised by lesion radius.

A sensible spatial proxy is a radial distance map inside the lesion:
- compute centroid
- compute max radius to boundary
- for each lesion pixel: distance to centroid / max_radius

Method still working with public, automatic features. No clinicians required.

Answering question: Do XAI methods attend to known risk factors (ABCD)?

The evaluation is transformed from pure geometry (mask overlap) to semi-semantic interpretability:  a richer, more original practicum.